In [12]:
from dotenv import load_dotenv
import os
import getpass

# Load environment variables from .env file
load_dotenv()

# Get Neo4j and LangChain credentials
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
LANGCHAIN_API_KEY = os.getenv("LANGCHAIN_API_KEY")

# Set LangChain tracing (if API key exists)
if LANGCHAIN_API_KEY:
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
    os.environ["LANGCHAIN_API_KEY"] = LANGCHAIN_API_KEY  # Not needed, but okay

# Set OpenAI API key (ask only if not in .env)
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

print("Environment variables loaded successfully!")

Environment variables loaded successfully!


In [13]:
from langchain.vectorstores.neo4j_vector import Neo4jVector
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.retrievers import BaseRetriever
from typing import List
from pydantic import BaseModel, Field

In [19]:
vector_index = Neo4jVector.from_existing_graph(
    OpenAIEmbeddings(),
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    index_name="index_for_Product",
    node_label="Product",
    text_node_properties=[
        "name", "date", "sensorType", "organization", "orbitType",
        "spatialExtent", "instrument", "isoStandard", "visualizationUrl",
        "referenceSystem", "platform", "compositeType"
    ],
    embedding_node_property="embedding",
)

# ✅ Custom Retriever with Score Adjustments
class CustomRetriever(BaseRetriever, BaseModel):
    """Custom retriever that fetches relevant documents from Neo4j and scores them."""
    
    vector_index: Neo4jVector = Field(...)

    def _get_relevant_documents(self, query: str) -> List[Document]:
        """Retrieve and score documents from the vector index."""
        docs, scores = zip(*self.vector_index.similarity_search_with_score(query))
        for doc, score in zip(docs, scores):
            doc.metadata["score"] = score  # Add scores to metadata
        return docs

# Instantiate retriever
retriever = CustomRetriever(vector_index=vector_index)

# ✅ Update scores for retrieved documents
def update_scores(docs):
    for doc in docs:
        doc.metadata["score"] *= 10  # Scale score
        doc.page_content += f"\nScore: {doc.metadata['score']}"
    return docs

# ✅ Define LLM Prompt Template
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

# ✅ LLM Model
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

# ✅ LangChain Processing Pipeline
graph_chain = (
    {"context": retriever | update_scores, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# ✅ Example Question
question = "What dataset has same sensor types mention sensor types and  what is it, then mension all other  sensor types?"

# Run the pipeline and get the answer
response = graph_chain.invoke(question)

# ✅ Output Answer
print(response)

The dataset "Sentinel-2 L2A Maja" has the same sensor type as "EnMAP L2A HSI Products," which is Optical. The other sensor types mentioned in the metadata are not specified in the provided context.
